In [ ]:
# ============================================================
# DeepLure — Color-Invariant Saree Design Recognition
# Kaggle Notebook (Python Script — paste into Kaggle cells)
# ============================================================
# This is the notebook source as a single runnable Python file.
# On Kaggle: split at each "# ── CELL" comment into separate cells.
# Accelerator: GPU T4 x2 (or P100) recommended.
# ============================================================

# ── CELL 1 — Install dependencies ────────────────────────────
"""
!pip install -q timm faiss-gpu pillow --upgrade
"""

# ── CELL 2 — Imports & seeds ──────────────────────────────────
from __future__ import annotations

import io
import json
import math
import os
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageOps
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms as T


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── CELL 3 — Dataset & paths ─────────────────────────────────
# Kaggle dataset: https://www.kaggle.com/datasets/div456/indian-saree-patterns
# Mount: /kaggle/input/indian-saree-patterns/
# If DeepLure Drive corpus is available, mount to /kaggle/input/deeplure-saree/

DATA_PATHS = [
    Path("/kaggle/input/indian-saree-patterns"),
    Path("/kaggle/input/deeplure-saree"),
    Path("data/raw"),  # local fallback
]

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
OUTPUT_DIR = Path("/kaggle/working")
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

# Hyper-parameters
IMAGE_SIZE   = 128
EMBEDDING_DIM = 256
BACKBONE     = "resnet18"     # swap to efficientnet_b0 or convnext_atto
GEM_P        = 3.0
TEMPERATURE  = 0.07
EPOCHS       = 12
P_CLASSES    = 8              # classes per batch
K_PER_CLASS  = 4              # images per class
N_BATCHES    = 150            # batches per epoch
LR           = 1e-4
WEIGHT_DECAY = 1e-4
COLOR_MODE   = "full"         # none | standard | recolor | full

# ── CELL 4 — Recoloring engine ───────────────────────────────

PALETTES = {
    "red":    [(180,20,30),  (240,180,160)],
    "blue":   [(20,40,150),  (180,210,240)],
    "green":  [(20,120,50),  (200,230,180)],
    "purple": [(90,30,140),  (220,190,240)],
    "gold":   [(180,140,40), (250,230,170)],
    "teal":   [(10,110,120), (180,230,230)],
}

def _to_np(img): return np.asarray(img.convert("RGB"), dtype=np.float32)/255.
def _to_pil(a): return Image.fromarray(np.clip(a*255,0,255).astype(np.uint8))

def rgb_to_hsv(rgb):
    r,g,b = rgb[...,0],rgb[...,1],rgb[...,2]
    mx=np.max(rgb,-1); mn=np.min(rgb,-1); v=mx
    s=np.where(mx==0,0,(mx-mn)/np.clip(mx,1e-8,None))
    rc=(mx-r)/np.clip(mx-mn,1e-8,None); gc=(mx-g)/np.clip(mx-mn,1e-8,None); bc=(mx-b)/np.clip(mx-mn,1e-8,None)
    h=np.zeros_like(mx)
    h=np.where((mx==r)&(mx!=mn),(bc-gc),h)
    h=np.where((mx==g)&(mx!=mn),2.+(rc-bc),h)
    h=np.where((mx==b)&(mx!=mn),4.+(gc-rc),h)
    h=(h/6.)%1.; h=np.where(mx==mn,0.,h)
    return np.stack([h,s,v],-1)

def hsv_to_rgb(hsv):
    h,s,v=hsv[...,0],hsv[...,1],hsv[...,2]
    i=np.floor(h*6.).astype(np.int32); f=h*6.-i
    p=v*(1.-s); q=v*(1.-f*s); t_=v*(1.-(1.-f)*s); i=i%6
    rgb=np.zeros_like(hsv)
    for k,c in enumerate([np.stack([v,t_,p],-1),np.stack([q,v,p],-1),
                           np.stack([p,v,t_],-1),np.stack([p,q,v],-1),
                           np.stack([t_,p,v],-1),np.stack([v,p,q],-1)]):
        rgb[i==k]=c[i==k]
    return rgb

def lum(rgb): return 0.2126*rgb[...,0]+0.7152*rgb[...,1]+0.0722*rgb[...,2]

def hue_shift(img, shift):
    rgb=_to_np(img); hsv=rgb_to_hsv(rgb); hsv[...,0]=(hsv[...,0]+shift)%1.
    return _to_pil(hsv_to_rgb(hsv))

def to_gray(img):
    rgb=_to_np(img); y=lum(rgb); return _to_pil(np.stack([y,y,y],-1))

def palette_map(img, name):
    pal=np.array(PALETTES[name],dtype=np.float32)/255.
    rgb=_to_np(img); y=lum(rgb)
    edges=np.abs(np.gradient(y)[0])+np.abs(np.gradient(y)[1])
    edges=np.clip(edges/(edges.max()+1e-8),0,1)
    bins=np.linspace(0,1,len(pal)+1); out=np.zeros_like(rgb)
    for i,c in enumerate(pal):
        mask=(y>=bins[i])&(y<bins[i+1] if i<len(pal)-1 else True); out[mask]=c
    y_out=lum(out); sc=(y+1e-3)/(y_out+1e-3)
    out=np.clip(out*sc[...,None],0,1)
    out=out*(1-.35*edges[...,None])+rgb*(.35*edges[...,None])
    return _to_pil(out)

def random_colorway(img, rng=None):
    rng=rng or random
    names=list(PALETTES.keys())+["gray","hue"]
    name=rng.choice(names)
    if name in PALETTES: return palette_map(img,name)
    if name=="gray": return to_gray(img)
    return hue_shift(img, rng.uniform(0.1,0.9))

# ── CELL 5 — Transforms ──────────────────────────────────────

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

class RandomRecolor:
    def __init__(self, p=0.75): self.p=p
    def __call__(self, img):
        return random_colorway(img) if random.random()<self.p else img

class RandomHSV:
    def __init__(self, p=0.5): self.p=p
    def __call__(self, img):
        if random.random()>self.p: return img
        arr=np.asarray(img).astype(np.float32)/255.
        hsv=rgb_to_hsv(arr)
        hsv[...,0]=(hsv[...,0]+np.random.uniform(-.2,.2))%1.
        hsv[...,1]=np.clip(hsv[...,1]*np.random.uniform(.4,1.6),0,1)
        hsv[...,2]=np.clip(hsv[...,2]*np.random.uniform(.7,1.3),0,1)
        return Image.fromarray(np.clip(hsv_to_rgb(hsv)*255,0,255).astype(np.uint8))

class MaybeGray:
    def __init__(self, p=0.12): self.p=p
    def __call__(self, img):
        return ImageOps.grayscale(img).convert("RGB") if random.random()<self.p else img

def train_tfm(sz=IMAGE_SIZE):
    return T.Compose([
        T.RandomResizedCrop(sz, scale=(.6,1.), ratio=(.85,1.15)),
        T.RandomHorizontalFlip(.5),
        T.RandomAffine(8,translate=(.05,.05),scale=(.95,1.05),shear=4),
        T.ColorJitter(.35,.35,.5,.15),
        RandomHSV(.45),
        MaybeGray(.12),
        RandomRecolor(.75),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

def eval_tfm(sz=IMAGE_SIZE):
    return T.Compose([
        T.Resize(int(sz*1.14)),
        T.CenterCrop(sz),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

# ── CELL 6 — Dataset ─────────────────────────────────────────

def discover_images(root):
    return sorted(p for p in Path(root).rglob("*") if p.suffix.lower() in IMAGE_EXTS)

class SareeDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths=paths; self.labels=labels; self.transform=transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img=Image.open(self.paths[idx]).convert("RGB")
        if self.transform: img=self.transform(img)
        return img, self.labels[idx]

def build_dataset(data_roots, train_frac=.70, val_frac=.10, seed=SEED):
    """Discover images, assign class from parent-folder name, split."""
    all_paths=[]; all_classes=[]
    for root in data_roots:
        root=Path(root)
        if not root.exists(): continue
        imgs=discover_images(root)
        all_paths.extend(imgs)
        all_classes.extend([p.parent.name for p in imgs])
        print(f"  {root}: {len(imgs)} images")
    if not all_paths: raise RuntimeError("No images found in any data path")

    cls_set=sorted(set(all_classes))
    cls_map={c:i for i,c in enumerate(cls_set)}
    labels=[cls_map[c] for c in all_classes]

    # Group-level split
    groups=defaultdict(list)
    for i,(p,l) in enumerate(zip(all_paths,labels)):
        groups[l].append(i)
    rng=np.random.default_rng(seed)
    gids=list(groups.keys()); rng.shuffle(gids)
    n=len(gids); n_tr=int(round(train_frac*n)); n_va=int(round(val_frac*n))
    tr_idx=[]; va_idx=[]; te_idx=[]
    for i,g in enumerate(gids):
        if i<n_tr: tr_idx.extend(groups[g])
        elif i<n_tr+n_va: va_idx.extend(groups[g])
        else: te_idx.extend(groups[g])

    print(f"Classes: {len(cls_set)}  Train: {len(tr_idx)}  Val: {len(va_idx)}  Test: {len(te_idx)}")
    return all_paths, labels, tr_idx, va_idx, te_idx, len(cls_set)

# ── CELL 7 — Balanced sampler ─────────────────────────────────

class BalancedSampler:
    def __init__(self, labels, idxs, p=P_CLASSES, k=K_PER_CLASS, n_batches=N_BATCHES):
        self.p=p; self.k=k; self.n=n_batches
        c2i=defaultdict(list)
        for idx in idxs: c2i[labels[idx]].append(idx)
        self.c2i=c2i; self.classes=list(c2i.keys())
    def __iter__(self):
        for _ in range(self.n):
            cls=random.sample(self.classes, min(self.p,len(self.classes)))
            batch=[]
            for c in cls: batch.extend(random.choices(self.c2i[c], k=self.k))
            yield batch
    def __len__(self): return self.n

# ── CELL 8 — Model ───────────────────────────────────────────

class GeM(nn.Module):
    def __init__(self, p=3., eps=1e-6):
        super().__init__(); self.p=nn.Parameter(torch.ones(1)*p); self.eps=eps
    def forward(self, x):
        return F.avg_pool2d(x.clamp(self.eps).pow(self.p),(x.size(-2),x.size(-1))).pow(1./self.p).flatten(1)

class SareeNet(nn.Module):
    def __init__(self, backbone=BACKBONE, dim=EMBEDDING_DIM, gem_p=GEM_P, pretrained=True):
        super().__init__()
        if backbone=="resnet18":
            net=models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
            feat_dim=net.fc.in_features
            self.features=nn.Sequential(*list(net.children())[:-2])
        elif backbone=="efficientnet_b0":
            net=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT if pretrained else None)
            feat_dim=net.classifier[1].in_features; self.features=net.features
        else:
            import timm
            net=timm.create_model(backbone,pretrained=pretrained,num_classes=0,global_pool="")
            feat_dim=net.num_features; self.features=net
        self.pool=GeM(p=gem_p)
        self.proj=nn.Sequential(
            nn.Linear(feat_dim,feat_dim), nn.BatchNorm1d(feat_dim), nn.ReLU(True),
            nn.Linear(feat_dim,dim))

    def forward(self, x, norm=True):
        z=self.proj(self.pool(self.features(x)))
        return F.normalize(z,2,1) if norm else z

def count_params(m):
    tot=sum(p.numel() for p in m.parameters())
    tr=sum(p.numel() for p in m.parameters() if p.requires_grad)
    return tot,tr

# ── CELL 9 — SupCon loss ─────────────────────────────────────

class SupConLoss(nn.Module):
    def __init__(self, tau=TEMPERATURE):
        super().__init__(); self.tau=tau
    def forward(self, emb, labels):
        z=F.normalize(emb,2,1); labels=labels.view(-1)
        sim=z@z.T/self.tau
        mask=torch.ones_like(sim)-torch.eye(sim.size(0),device=sim.device)
        sim=sim-1e9*(1-mask)
        eq=labels.unsqueeze(0).eq(labels.unsqueeze(1)).float()
        pos=eq*mask; cnt=pos.sum(1); valid=cnt>0
        if not valid.any(): return emb.new_zeros(())
        lp=sim-torch.logsumexp(sim,1,keepdim=True)
        loss=-(pos*lp).sum(1)/cnt.clamp(min=1.)[valid]
        return loss.mean()

# ── CELL 10 — Training ───────────────────────────────────────

def train(model, loader, crit, opt, scaler, clip=1.0):
    model.train(); tot=0; steps=0
    for imgs,lbls in loader:
        imgs,lbls=imgs.to(DEVICE),lbls.to(DEVICE)
        opt.zero_grad()
        with torch.autocast(DEVICE.type, enabled=scaler is not None):
            loss=crit(model(imgs,norm=False),lbls)
        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),clip)
            scaler.step(opt); scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),clip)
            opt.step()
        tot+=loss.item(); steps+=1
    return tot/max(steps,1)

@torch.inference_mode()
def recall_at_k(model, ds, ks=(1,5,10), bs=64):
    loader=DataLoader(ds,bs,shuffle=False,num_workers=2)
    Zs,Ls=[],[]
    model.eval()
    for x,l in loader:
        Zs.append(model(x.to(DEVICE)).cpu()); Ls.append(l)
    Z=F.normalize(torch.cat(Zs),2,1); L=torch.cat(Ls)
    sim=Z@Z.T; sim.fill_diagonal_(-2.)
    out={}
    for k in ks:
        top=sim.topk(min(k,sim.size(1)),dim=1).indices
        out[f"R@{k}"]=float(L[top].eq(L[:,None]).any(1).float().mean())
    return out

def run_training():
    """Full training loop — call this cell to start training."""
    # 1. Dataset
    print("Discovering images…")
    all_paths,labels,tr_idx,va_idx,te_idx,n_cls=build_dataset(DATA_PATHS)

    tr_ds=SareeDataset([all_paths[i] for i in tr_idx],[labels[i] for i in tr_idx],train_tfm())
    va_ds=SareeDataset([all_paths[i] for i in va_idx],[labels[i] for i in va_idx],eval_tfm())
    te_ds=SareeDataset([all_paths[i] for i in te_idx],[labels[i] for i in te_idx],eval_tfm())

    sampler=BalancedSampler(labels,tr_idx)
    loader=DataLoader(tr_ds,batch_sampler=sampler,num_workers=2,pin_memory=True)

    # 2. Model
    model=SareeNet(BACKBONE,EMBEDDING_DIM,GEM_P).to(DEVICE)
    tot,tr=count_params(model)
    print(f"Params: {tot:,} total / {tr:,} trainable")

    crit=SupConLoss(TEMPERATURE)
    opt=optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    sched=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS,eta_min=LR*.01)
    scaler=torch.cuda.amp.GradScaler() if DEVICE.type=="cuda" else None

    # 3. Loop
    best_r1,history=-1,[]
    for ep in range(1,EPOCHS+1):
        t0=time.time()
        loss=train(model,loader,crit,opt,scaler)
        sched.step()
        if va_ds and len(va_ds)>1:
            metrics=recall_at_k(model,va_ds)
            r1=metrics["R@1"]
            if r1>best_r1:
                best_r1=r1
                torch.save({"state_dict":model.state_dict(),"meta":{
                    "backbone":BACKBONE,"embedding_dim":EMBEDDING_DIM,
                    "image_size":IMAGE_SIZE,"temperature":TEMPERATURE,
                    "color_mode":COLOR_MODE,"mode":"trained"
                }}, MODEL_DIR/"best_model.pt")
            print(f"Ep {ep:3d}/{EPOCHS}  loss={loss:.4f}  "+" ".join(f"{k}={v:.4f}" for k,v in metrics.items())+f"  {time.time()-t0:.0f}s")
            history.append({"epoch":ep,"loss":round(loss,4),**{k:round(v,4) for k,v in metrics.items()}})
        else:
            print(f"Ep {ep:3d}/{EPOCHS}  loss={loss:.4f}  {time.time()-t0:.0f}s")
            history.append({"epoch":ep,"loss":round(loss,4)})

    print(f"\n✓ Training done. Best R@1={best_r1:.4f}")
    return model, history, te_ds, labels, te_idx, all_paths

# ── CELL 11 — Run training ───────────────────────────────────
model, history, te_ds, labels, te_idx, all_paths = run_training()

# ── CELL 12 — Evaluation ─────────────────────────────────────

def mean_ap(Z, L):
    sim=Z@Z.T; np.fill_diagonal(sim,-2.)
    aps=[]
    for i in range(len(L)):
        order=np.argsort(-sim[i])
        rel=(L[order]==L[i]).astype(float)
        rel=rel[order!=i]
        if rel.sum()==0: continue
        cs=np.cumsum(rel); prec=cs/(np.arange(len(rel))+1)
        aps.append((prec*rel).sum()/rel.sum())
    return float(np.mean(aps)) if aps else 0.

def roc_eer(Z, L, n_pairs=50_000):
    rng=np.random.default_rng(42); n=len(Z)
    ia=rng.integers(0,n,n_pairs); ib=rng.integers(0,n,n_pairs)
    m=ia!=ib; ia,ib=ia[m],ib[m]
    sims=(Z[ia]*Z[ib]).sum(1); same=(L[ia]==L[ib]).astype(int)
    thrs=np.linspace(-1,1,200); tprs,fprs=[],[]
    for t in thrs:
        p=(sims>=t).astype(int)
        tp=((p==1)&(same==1)).sum(); fp=((p==1)&(same==0)).sum()
        fn=((p==0)&(same==1)).sum(); tn=((p==0)&(same==0)).sum()
        tprs.append(tp/max(tp+fn,1)); fprs.append(fp/max(fp+tn,1))
    auc=float(-np.trapz(tprs,fprs))
    frrs=1-np.array(tprs); diff=np.abs(np.array(fprs)-frrs)
    ei=np.argmin(diff); eer=(fprs[ei]+frrs[ei])/2
    return {"auc":round(auc,4),"eer":round(float(eer),4),"eer_thr":round(float(thrs[ei]),4)}

def evaluate(model, ds, name="test"):
    loader=DataLoader(ds,64,shuffle=False,num_workers=2)
    Zs,Ls=[],[]
    model.eval()
    with torch.inference_mode():
        for x,l in loader:
            Zs.append(model(x.to(DEVICE)).cpu().numpy()); Ls.append(l.numpy())
    Z=np.vstack(Zs); Z/=np.linalg.norm(Z,axis=1,keepdims=True)+1e-8
    L=np.concatenate(Ls)
    print(f"\n── {name.upper()} SET ──")
    r=recall_at_k(model,ds)
    for k,v in r.items(): print(f"  {k}: {v:.4f}")
    mAP=mean_ap(Z,L); print(f"  mAP: {mAP:.4f}")
    vf=roc_eer(Z,L); print(f"  AUC: {vf['auc']:.4f}  EER: {vf['eer']:.4f}  EER-thr: {vf['eer_thr']:.4f}")
    tot,_=count_params(model); print(f"  Params: {tot:,}  EmbDim: {EMBEDDING_DIM}")
    return {**r,"mAP":round(mAP,4),**vf}

results = evaluate(model, te_ds, "test")
print("\nFull results:", json.dumps(results, indent=2))

# ── CELL 13 — Save checkpoint & results ─────────────────────
(MODEL_DIR/"results.json").write_text(json.dumps({
    "backbone":BACKBONE,"embedding_dim":EMBEDDING_DIM,
    "image_size":IMAGE_SIZE,"temperature":TEMPERATURE,
    "color_mode":COLOR_MODE,"metrics":results
},indent=2))
print("Saved to", MODEL_DIR)

# ── CELL 14 — Inference demo ─────────────────────────────────

def embed_image(model, img_path_or_pil):
    tfm=eval_tfm()
    if isinstance(img_path_or_pil, (str, Path)):
        img=Image.open(img_path_or_pil).convert("RGB")
    else:
        img=img_path_or_pil.convert("RGB")
    t=tfm(img).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        return model(t).cpu().numpy()[0]

def demo_verify(model, path_a, path_b, threshold=0.5):
    za=embed_image(model, path_a); zb=embed_image(model, path_b)
    sim=float(np.dot(za,zb)); same=sim>=threshold
    print(f"Similarity: {sim:.4f}  →  {'SAME DESIGN ✅' if same else 'DIFFERENT ❌'}")
    return sim, same

print("\nDemo complete — model is ready for inference.")
print(f"Checkpoint: {MODEL_DIR/'best_model.pt'}")
